# 3.3 Chains — Connecting Components for Complex Workflows

## Playground Notebook

Chains are LangChain's mechanism for connecting components into **complete, reusable pipelines**. Think of them as assembly lines — raw materials (user input) enter one end, pass through specialized stations, and a finished product emerges.

| Topic | What You'll Learn |
|-------|-------------------|
| **Sequential Chains** | Linear pipelines with context accumulation |
| **Parallel Chains** | Executing branches simultaneously |
| **Conditional Chains** | Routing input to different paths |
| **Document Processing** | Stuff, Map-Reduce, Refine, Map-Rerank strategies |
| **Conversational Chains** | Memory-integrated multi-turn dialogue |
| **Transformation Chains** | Custom pre/post-processing with RunnableLambda |
| **Chain Composition** | Nesting chains and self-correcting patterns |

### Chain Categories

| Category | Flow | Example |
|----------|------|---------|
| **Sequential** | A → B → C | prompt → model → parser |
| **Parallel** | A ├→ B, C → merge | Retrieve docs AND classify query |
| **Conditional** | A → if/else → B or C | Route by question type |
| **Map-Reduce** | map(each) → reduce(all) | Summarize many documents |
| **Recursive** | A → B → check → loop | Self-correcting outputs |

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [13]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch,
)
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. Sequential Chains — Linear Pipelines

The most common pattern. Data flows through a linear pipeline where each step transforms it for the next.

```
Input → Step A → Step B → Step C → Output
```

### Experiment 1A: The Basic Chain — prompt | model | parser

In [14]:
# The fundamental building block of ALL chains

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant. Only answer in 2 sentences max."),
    ("human", "{question}")
])

parser = StrOutputParser()

# prompt → model → parser
basic_chain = prompt | llm | parser

result = basic_chain.invoke({"question": "What are chains in LangChain?"})
print(f"Type: {type(result).__name__}")  # str
display(Markdown(result))

Type: TextAccessor


In LangChain, a chain is a sequence of tasks that can be executed together to achieve a specific goal. It's a way to compose multiple NLP tasks into a single workflow, allowing for efficient and modular execution of complex tasks.

### Experiment 1B: Multi-Step Sequential Chain

In [15]:
# Multi-step: each chain step is its own prompt → model → parser
# Output of each step feeds into the next

# Step 1: Generate an idea
idea_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You generate creative app ideas. Respond with ONLY the app name and a 1-2 sentence description. No bullet points, no headers, no explanations."),
        ("human", "Give me an app idea for: {domain}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Create a tagline
tagline_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You write catchy taglines. Respond with ONLY one tagline sentence. Nothing else."),
        ("human", "Write a tagline for this app idea: {idea}")
    ])
    | llm | StrOutputParser()
)

# Step 3: List key features
features_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You list product features. Respond with ONLY 3 bullet points, one line each. No intro, no outro."),
        ("human", "List 3 key features for: {idea}")
    ])
    | llm | StrOutputParser()
)

# Run sequentially — output of each step feeds into the next
domain = "education"

print(f"Domain: {domain}")
print(f"\n{'=' * 50}")
print("Step 1 — App Idea:")
print(f"{'=' * 50}")
idea = idea_chain.invoke({"domain": domain})
print(idea)

print(f"\n{'=' * 50}")
print("Step 2 — Tagline:")
print(f"{'=' * 50}")
tagline = tagline_chain.invoke({"idea": idea})
print(tagline)

print(f"\n{'=' * 50}")
print("Step 3 — Key Features:")
print(f"{'=' * 50}")
features = features_chain.invoke({"idea": idea})
display(Markdown(features))

Domain: education

Step 1 — App Idea:
MindPal

Step 2 — Tagline:
"Find your inner calm, one thought at a time."

Step 3 — Key Features:


• **Personalized cognitive coaching**: MindPal provides personalized cognitive training and coaching to help users improve their focus, attention, mental clarity, and overall well-being.

• **AI-driven progress tracking**: The app uses AI-powered algorithms to track user progress, identifying areas of strength and weakness, and providing tailored recommendations for improvement.

• **Mindfulness and meditation exercises**: MindPal offers a range of guided mindfulness and meditation exercises to help users reduce stress, increase relaxation, and enhance their mental clarity.

### Experiment 1C: Context Accumulation with `assign()`

In multi-step chains, later steps often need outputs from **earlier** steps — not just the previous one. `RunnablePassthrough.assign()` accumulates context so every step sees everything before it.

```
Start:        {"topic": "AI"}
After Step 1: {"topic": "AI", "summary": "..."}
After Step 2: {"topic": "AI", "summary": "...", "key_points": "..."}
After Step 3: {"topic": "AI", "summary": "...", "key_points": "...", "verdict": "..."}
```

In [16]:
# Context accumulation — each step sees ALL previous outputs

# Step 1: Summarize
summarize_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You write concise technical notes. "
         "Summarize the topic in plain prose. "
         "Your response must be under 40 words total."),
        ("human", "Summarize: {topic}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Extract key points (uses topic + summary)
keypoints_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Based on this summary: {summary}\n\n"
         "Extract 3 key points. One short line per point, under 10 words each.\n"
         "Use this exact format:\n"
         "1. First point here\n"
         "2. Second point here\n"
         "3. Third point here\n\n"
         "Write nothing else. Only 3 lines."),
        ("human", "Topic: {topic}")
    ])
    | llm | StrOutputParser()
)

# Step 3: Generate verdict (uses topic + summary + key_points)
verdict_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Summary: {summary}\n"
         "Key Points: {key_points}\n\n"
         "Write a single-sentence verdict on why this topic matters. "
         "Your response must be under 25 words. No lists, no headings."),
        ("human", "Topic: {topic}")
    ])
    | llm | StrOutputParser()
)

# Compose with progressive context accumulation
analysis_pipeline = (
    RunnablePassthrough.assign(summary=summarize_chain)
    | RunnablePassthrough.assign(key_points=keypoints_chain)
    | RunnablePassthrough.assign(verdict=verdict_chain)
)

result = analysis_pipeline.invoke({"topic": "Chains in LangChain"})

print("PROGRESSIVE CONTEXT ACCUMULATION")
print("=" * 50)
print(f"\nTopic: {result['topic']}")
print(f"\nSummary:\n  {result['summary']}")
print(f"\nKey Points:\n  {result['key_points']}")
print(f"\nVerdict:\n  {result['verdict']}")

PROGRESSIVE CONTEXT ACCUMULATION

Topic: Chains in LangChain

Summary:
  Chains are a fundamental data structure in LangChain, a framework for building custom workflows and pipelines in Python. A Chain represents a sequence of tasks or operations that must be executed in a specific order to achieve a desired outcome. Chains can be used to model complex workflows, such as data processing pipelines, natural language processing workflows, or machine learning pipelines, among others.

Key Points:
  1. Chains model complex workflows.
2. Represent a sequence of tasks or operations.
3. Execute tasks in a specific order to achieve an outcome.

Verdict:
  In LangChain, a Chain is a fundamental data structure that represents a sequence of tasks or operations that must be executed in a specific order to achieve a desired outcome. It's a powerful tool for building custom workflows and pipelines.

Here are some key points about Chains in LangChain:

A Chain can be thought of as a directed graph whe

---

## 2. Parallel Chains — Execute Branches Simultaneously

Run multiple independent operations on the same input **at the same time**.

```
         ┌→ Branch A →┐
Input →  ├→ Branch B →├→ Merged Output
         └→ Branch C →┘
```

### Performance Impact

| Approach | 3 LLM calls × 2s each |
|----------|----------------------|
| Sequential | ~6 seconds |
| Parallel | ~2 seconds |

### Experiment 2A: Parallel Analysis — Multiple Aspects at Once

In [17]:
# Analyze a topic from 3 perspectives simultaneously

def make_chain(system_prompt):
    return (
        ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{topic}")
        ])
        | llm | StrOutputParser()
    )

# Three independent analysis branches — each with tight word limits
parallel_analysis = RunnableParallel(
    summary=make_chain(
        "You write concise technical summaries. "
        "Summarize in plain prose. Under 30 words."
    ),
    pros=make_chain(
        "List exactly 2 advantages. One line each, under 12 words per line. "
        "Format:\n- Advantage one\n- Advantage two\n"
        "Write nothing else."
    ),
    cons=make_chain(
        "List exactly 2 limitations. One line each, under 12 words per line. "
        "Format:\n- Limitation one\n- Limitation two\n"
        "Write nothing else."
    ),
)

# All three run at the same time
start = time.time()
result = parallel_analysis.invoke({"topic": "Using chains in LangChain for production apps"})
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f}s (3 branches ran in parallel!)\n")
print("SUMMARY:")
print(f"  {result['summary']}")
print(f"\nPROS:")
print(f"  {result['pros']}")
print(f"\nCONS:")
print(f"  {result['cons']}")

Completed in 37.47s (3 branches ran in parallel!)

SUMMARY:
  Chains are a powerful feature in LangChain, an open-source framework for building scalable and modular text processing pipelines.

**What are Chains?**

In LangChain, a chain is a sequence of tasks that are executed one after another. Each task can be a function that takes input data and produces output data. Chains allow you to break down complex text processing workflows into smaller, reusable components, making it easier to manage complexity and improve maintainability.

**Using Chains in Production Apps**

Chains can be used in production apps to:

1. **Automate workflows**: Chains can automate repetitive tasks, such as data preprocessing, feature extraction, or model training.
2. **Improve scalability**: By breaking down complex workflows into smaller chains, you can scale individual components independently, reducing the overall load on your application.
3. **Enhance modularity**: Chains enable modular code organizatio

### Experiment 2B: Parallel → Sequential — Merge Then Synthesize

In [20]:
# Parallel branches → merge results → synthesize into a final report

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You write balanced technical assessments. "
     "Given this analysis, write a brief assessment in plain prose.\n\n"
     "Summary: {summary}\n"
     "Pros: {pros}\n"
     "Cons: {cons}\n\n"
     "Your response must be under 60 words. No bullet points, no headings."),
    ("human", "Provide your assessment.")
])

# Full pipeline: parallel analysis → synthesis
full_analysis = parallel_analysis | synthesis_prompt | llm | StrOutputParser()

result = full_analysis.invoke({"topic": "Map-Reduce chains for document processing"})

print("SYNTHESIZED ASSESSMENT:")
print("=" * 50)
display(Markdown(result))

SYNTHESIZED ASSESSMENT:


**Assessment: Sentence-BERT (sBERT)**

**What is it?**
Sentence-BERT (sBERT) is a type of deep learning model that transforms sentences into numerical vectors, capturing their semantic meaning for various natural language processing (NLP) tasks.

**How does it work?**

1. **Input**: A sequence of tokens (e.g., words or subwords) is fed into the sBERT model.
2. **Transformation**: The model applies a multi-layer bidirectional transformer encoder architecture to generate a fixed-length vector representation for each sentence.
3. **Output**: The resulting vector representation captures the semantic meaning of the input sentence.

**Benefits**
sBERT offers several advantages:

* **Improved semantic understanding**: Captures the meaning of entire sentences, enabling more accurate context understanding.
* **Increased representation capacity**: Learns higher-dimensional representations than traditional word embeddings.
* **Efficient computation**: Faster to compute than traditional NLP tasks relying on word embeddings.

**Common applications**
sBERT is commonly used for:

* Text classification
* Sentiment analysis
* Question answering
* Named entity recognition
* Machine translation

**Pros and Cons**

Pros: Improved semantic understanding, increased representation capacity, efficient computation.

Cons: Limited by data quality and quantity, may not perform well on tasks requiring complex syntax or semantics.

### Experiment 2C: Sequential vs Parallel Timing

In [21]:
# Direct comparison: sequential vs parallel execution

chain_a = make_chain("You write concise definitions. Define in one sentence, under 20 words.")
chain_b = make_chain("Name one key benefit in a single sentence. Under 15 words.")
chain_c = make_chain("Name one limitation in a single sentence. Under 15 words.")

topic = {"topic": "LangChain chains"}

# Sequential
start = time.time()
r1 = chain_a.invoke(topic)
r2 = chain_b.invoke(topic)
r3 = chain_c.invoke(topic)
seq_time = time.time() - start

# Parallel
start = time.time()
parallel = RunnableParallel(a=chain_a, b=chain_b, c=chain_c)
results = parallel.invoke(topic)
par_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Parallel:   {par_time:.2f}s")
print(f"Speedup:    {seq_time/par_time:.1f}x")

# NOTE: With local Ollama on a single GPU, parallel requests share the
# same GPU so you may see NO speedup (or even slowdown). True parallel
# speedup requires remote APIs (OpenAI, Anthropic) or multiple GPUs.
# The RunnableParallel pattern is correct — the bottleneck is hardware.

Sequential: 20.54s
Parallel:   19.02s
Speedup:    1.1x


---

## 3. Conditional Chains — Routing by Condition

Route input to different processing paths based on conditions.

```
Input → Condition Check
            ├→ True  → Chain A
            └→ False → Chain B
```

### Experiment 3A: Keyword-Based Routing with RunnableBranch

In [23]:
# Route to different chains based on keywords in the input

technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a precise software engineer writing quick technical answers. "
         "Use technical terms. Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)

simple_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You explain things to a 10-year-old using a single analogy. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[SIMPLE] {x}")
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful assistant who gives short, direct answers. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)

router = RunnableBranch(
    (lambda x: any(w in x["question"].lower() for w in ["code", "api", "implement", "function"]),
     technical_chain),
    (lambda x: any(w in x["question"].lower() for w in ["simple", "easy", "explain like", "beginner"]),
     simple_chain),
    general_chain  # default
)

test_questions = [
    "How do I implement a sequential chain in code?",
    "Explain chains in a simple way for a beginner.",
    "What are the benefits of using LangChain?",
]

for q in test_questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    print(router.invoke({"question": q}))


Q: How do I implement a sequential chain in code?
[TECHNICAL] In programming, a sequential chain is typically represented using a linked list or an array. Here's a simple example of implementing a sequential chain using a singly-linked list in Python:

```python
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class SequentialChain:
    def __init__(self):
        self.head = None

    def append(self, value):
        new_node = Node(value)
        if not self.head:
            self.head = new_node
        else:
            current = self.head
            while current.next:
                current = current.next
            current.next = new_node

    def print_chain(self):
        values = []
        current = self.head
        while current:
            values.append(current.value)
            current = current.next
        return values

# Create a sequential chain and append some values
chain = SequentialChain()
chain.append(1)
chai

### Experiment 3B: LLM-Based Routing (Classify → Route)

Instead of hardcoded keyword rules, use the **LLM itself** to classify and route.

In [24]:
# Step 1: LLM classifies the question
classifier = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a question classifier. "
         "Classify this question into exactly one category.\n"
         "Valid categories: technical, creative, general\n"
         "Respond with ONLY that one word. Nothing else."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
)

# Step 2: Add classification to input
classified = RunnablePassthrough.assign(category=classifier)

# Step 3: Route based on LLM classification
creative_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a creative storyteller. Use vivid imagery and metaphor. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[CREATIVE] {x}")
)

smart_router = RunnableBranch(
    (lambda x: "technical" in x["category"].lower(), technical_chain),
    (lambda x: "creative" in x["category"].lower(), creative_chain),
    general_chain
)

# Full pipeline: classify → route
smart_pipeline = classified | smart_router

test_qs = [
    "How do I build a conditional chain with LCEL?",
    "Write a metaphor about data pipelines.",
    "What is the capital of France?",
]

for q in test_qs:
    classified_result = classified.invoke({"question": q})
    print(f"\nQ: {q}")
    print(f"Classified: {classified_result['category'].strip()}")
    print(f"Answer: {smart_router.invoke(classified_result)}")


Q: How do I build a conditional chain with LCEL?
Classified: I couldn't find any information on "LCEL". Could you provide more context or clarify what LCEL stands for?

If you meant to ask about building a conditional chain using another model, please provide more details and I'll be happy to help.
Answer: [GENERAL] To build a conditional chain using Long Short-Term Echo Chamber (LCEL), follow these steps:

1. Preprocess your data by converting text into vectors.
2. Use an encoder to map text inputs to vector representations.
3. Build the echo chamber graph, where each node represents a state in the model and edges represent transitions between states based on input and context.
4. Initialize the starting node (context) with a random or learned initial state.
5. Iterate through your data using the following steps:
	* For each input, compute the next state by applying the transition probabilities based on the current state, input, and previous output.
	* Keep track of the most likely n

---

## 4. Document Processing Chains

When documents are too large for a single LLM context window, you need strategies to process them in parts.

| Strategy | Speed | Quality | Cross-Doc Synthesis | Best For |
|----------|-------|---------|--------------------|---------|
| **Stuff** | Fast | High (if fits) | Yes | Small doc sets |
| **Map-Reduce** | Fast (parallel) | Medium | Limited | Large collections |
| **Refine** | Slow (sequential) | Highest | Excellent | Detailed analysis |
| **Map-Rerank** | Fast (parallel) | Medium | None | Single-answer Q&A |

### Experiment 4A: Stuff Chain — Combine Everything Into One Prompt

In [25]:
# Simulated documents (in production, these come from a document loader)
documents = [
    "Sequential chains process data through a linear pipeline where each step transforms data for the next.",
    "Parallel chains execute multiple operations simultaneously on the same input, collecting results.",
    "Conditional chains route input to different processing paths based on runtime conditions.",
    "Map-Reduce chains process each document independently, then combine all results.",
]

# STUFF strategy: combine all docs into one prompt
def stuff_documents(docs: list[str]) -> str:
    return "\n\n".join(f"Document {i+1}: {doc}" for i, doc in enumerate(docs))

stuff_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions based ONLY on the provided documents. "
     "Be direct and specific. Your answer must be under 50 words.\n\n"
     "{context}"),
    ("human", "{question}")
])

stuff_chain = stuff_prompt | llm | StrOutputParser()

result = stuff_chain.invoke({
    "context": stuff_documents(documents),
    "question": "What is the difference between sequential and parallel chains?"
})

print("STUFF CHAIN — All docs in one prompt")
print("=" * 50)
display(Markdown(result))
print(f"\nWorks great when all docs fit in the context window!")

STUFF CHAIN — All docs in one prompt


Sequential chains process data through a linear pipeline where each step transforms data for the next, whereas Parallel chains execute multiple operations simultaneously on the same input, collecting results.

In other words:

* Sequential chains are like a row-by-row processing of data.
* Parallel chains are like processing all rows at once.


Works great when all docs fit in the context window!


### Experiment 4B: Map-Reduce Chain — Process Each, Then Combine

```
Phase 1 — Map (parallel):
    Doc 1 → Summarize → Summary 1
    Doc 2 → Summarize → Summary 2
    Doc 3 → Summarize → Summary 3

Phase 2 — Reduce:
    [Summary 1, 2, 3] → Combine → Final Output
```

In [26]:
# MAP phase: summarize each document independently
map_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You write one-line summaries. "
         "Summarize this document in a single sentence, under 20 words."),
        ("human", "{doc}")
    ])
    | llm | StrOutputParser()
)

# Map: process each doc (using batch for parallel execution)
print("PHASE 1 — MAP (parallel summaries)")
print("=" * 50)

start = time.time()
summaries = map_chain.batch([{"doc": doc} for doc in documents])
map_time = time.time() - start

for i, s in enumerate(summaries):
    print(f"  Doc {i+1}: {s}")

print(f"\n  Map time: {map_time:.2f}s ({len(documents)} docs in parallel)")

PHASE 1 — MAP (parallel summaries)
  Doc 1: This is the core concept of Transformer models, which revolutionized the field of natural language processing (NLP).
  Doc 2: Parallel chains process inputs concurrently to accelerate computation by executing multiple operations at once.
  Doc 3: Conditional chains are a type of routing mechanism that directs input to specific processing paths at runtime, allowing for dynamic decision-making and flexibility in software systems.
  Doc 4: Sentence transformer models process individual sentences independently before combining their outputs to produce a final representation for a entire document or text.

  Map time: 2.12s (4 docs in parallel)


In [27]:
# REDUCE phase: combine all summaries into a final synthesis
reduce_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You synthesize multiple summaries into one coherent paragraph. "
         "Combine the key ideas into a single paragraph. "
         "Your response must be under 60 words. No bullet points."),
        ("human", "Summaries:\n{combined_summaries}")
    ])
    | llm | StrOutputParser()
)

print("\nPHASE 2 — REDUCE (synthesize)")
print("=" * 50)

combined = "\n".join(f"- {s}" for s in summaries)
final = reduce_chain.invoke({"combined_summaries": combined})
display(Markdown(final))


PHASE 2 — REDUCE (synthesize)


The core concept of Transformer models is the parallelization of chains, which allows for concurrent execution of multiple operations on inputs, significantly accelerating computation. This is achieved through routing mechanisms like conditional chains, enabling dynamic decision-making and flexibility in software systems. Additionally, sentence transformer models process individual sentences independently before combining their outputs to produce a final representation, effectively capturing the meaning of entire documents or texts.

### Experiment 4C: Refine Chain — Sequential Refinement

Process documents in order, refining the answer with each new document.

```
Doc 1 → Generate initial answer
Doc 2 + previous answer → Refine
Doc 3 + previous answer → Refine
→ Final refined answer
```

In [28]:
# Initial chain — first document generates the base answer
initial_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You answer questions based on a single document. "
         "Be direct and factual. Your answer must be under 40 words."),
        ("human", "Document: {doc}\n\nQuestion: {question}")
    ])
    | llm | StrOutputParser()
)

# Refine chain — each subsequent doc refines the existing answer
refine_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You refine answers by incorporating new information. "
         "Keep your existing answer and add only new relevant facts. "
         "If the new document is not relevant, return the existing answer unchanged. "
         "Your refined answer must be under 60 words."),
        ("human",
         "Existing answer: {existing_answer}\n\n"
         "New document: {doc}\n\n"
         "Question: {question}")
    ])
    | llm | StrOutputParser()
)

question = "How do chains work in LangChain?"

print(f"Question: {question}")
print(f"Processing {len(documents)} documents sequentially...\n")

# Process first document
answer = initial_chain.invoke({"doc": documents[0], "question": question})
print(f"After Doc 1: {answer}")

# Refine with each subsequent document
for i, doc in enumerate(documents[1:], 2):
    answer = refine_chain.invoke({
        "existing_answer": answer,
        "doc": doc,
        "question": question
    })
    print(f"\nAfter Doc {i}: {answer}")

print(f"\n{'=' * 50}")
print("FINAL REFINED ANSWER:")
print(f"{'=' * 50}")
display(Markdown(answer))

Question: How do chains work in LangChain?
Processing 4 documents sequentially...

After Doc 1: In LangChain, chains are composed of sequential nodes that process data one after another, with each node transforming the output of the previous node. The chain is executed from start to finish, and the final output is generated by the last node in the chain.

After Doc 2: Existing answer: In LangChain, chains are composed of sequential nodes that process data one after another, with each node transforming the output of the previous node. The chain is executed from start to finish, and the final output is generated by the last node in the chain.

New information: However, LangChain also supports parallel execution through the use of parallel chains. In a parallel chain, multiple nodes are executed simultaneously on the same input, collecting their results and producing an output for each node. This allows for faster processing times and improved performance on computationally intensive task

Chains in LangChain work by sequentially executing a series of nodes that process data one after another. Each node transforms the output of the previous node, ultimately generating the final output through the last node in the chain. Additionally, LangChain supports parallel execution through parallel chains, where multiple nodes are executed simultaneously on the same input to improve processing speed and performance on computationally intensive tasks.

### Experiment 4D: Map-Rerank Chain — Find the Best Answer

Process each document independently, score each answer, return the **best one**.

```
Doc 1 → Answer + Score → 0.85
Doc 2 → Answer + Score → 0.62
Doc 3 → Answer + Score → 0.94  ← Winner
```

In [29]:
# Map-Rerank: score each answer and pick the best

score_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You answer questions based on a document and rate your confidence. "
         "Respond with ONLY valid JSON in this exact format:\n"
         '{{"answer": "your short answer here", "score": 85}}\n'
         "The answer must be under 20 words. Score from 0-100. "
         "No markdown, no explanation, only the JSON object."),
        ("human", "Document: {doc}\n\nQuestion: {question}")
    ])
    | llm | JsonOutputParser()
)

question = "What is the advantage of parallel chains?"

# Map: score each document's answer
print(f"Question: {question}\n")
scored_answers = []

for i, doc in enumerate(documents):
    try:
        result = score_chain.invoke({"doc": doc, "question": question})
        scored_answers.append(result)
        print(f"  Doc {i+1}: score={result.get('score', 'N/A')} | {result.get('answer', 'N/A')[:80]}...")
    except Exception as e:
        print(f"  Doc {i+1}: Parse error — {e}")

# Rerank: pick the highest-scoring answer
if scored_answers:
    best = max(scored_answers, key=lambda x: x.get("score", 0))
    print(f"\n{'=' * 50}")
    print(f"BEST ANSWER (score: {best.get('score', 'N/A')}):")
    print(f"{'=' * 50}")
    print(best.get("answer", "No answer"))

Question: What is the advantage of parallel chains?

  Doc 1: Parse error — Invalid json output: {"answer": "Faster processing with shared computations", score: 95}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
  Doc 2: score=90 | Improved efficiency by reducing processing time...
  Doc 3: score=90 | Increased throughput by reducing computation dependencies...
  Doc 4: score=90 | Parallelization improves scalability and reduces processing time...

BEST ANSWER (score: 90):
Improved efficiency by reducing processing time


---

## 5. Conversational Chains — Memory-Integrated Dialogue

LLMs are **stateless** — each call is independent. Conversational chains add memory to make them behave as stateful conversational partners.

### Memory Types

| Memory Type | How It Works | Best For |
|-------------|--------------|----------|
| **Buffer** | Stores all messages verbatim | Short conversations (<20 turns) |
| **Window** | Keeps last K message pairs | Medium conversations |
| **Token Buffer** | Keeps messages up to N tokens | Token-budget-aware apps |
| **Summary** | LLM summarizes older messages | Long conversations |
| **Summary Buffer** | Summary of old + verbatim recent | Best balance overall |

### Experiment 5A: Buffer Memory — Full History

In [30]:
# Manual buffer memory with MessagesPlaceholder

convo_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly AI tutor who gives short, direct answers. "
     "Your response must be under 30 words. No bullet points."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

convo_chain = convo_prompt | llm | StrOutputParser()

# Buffer memory — stores everything
history = []

def chat(user_input):
    """Send a message and update history."""
    response = convo_chain.invoke({"history": history, "input": user_input})
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# Multi-turn conversation
turns = [
    "My name is Alex and I'm learning about LangChain chains.",
    "What is the simplest type of chain?",
    "What's my name and what am I learning?",
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i}")
    print(f"  User: {msg}")
    print(f"  AI:   {chat(msg)}")

print(f"\nHistory size: {len(history)} messages")
print("Buffer memory stores everything — grows unbounded!")


Turn 1
  User: My name is Alex and I'm learning about LangChain chains.
  AI:   LangChain is a Python library that enables building custom workflows by chaining together existing components, like you're doing. What specific part of LangChain are you learning about?

Turn 2
  User: What is the simplest type of chain?
  AI:   The simplest type of chain in LangChain is an `Item`, which represents a single item or data point being processed within a workflow.

Turn 3
  User: What's my name and what am I learning?
  AI:   Your name is Alex, and you're learning about LangChain chains.

History size: 6 messages
Buffer memory stores everything — grows unbounded!


### Experiment 5B: Window Memory — Last K Turns Only

In [31]:
# Window memory — only keep the last K exchanges

full_history = []
WINDOW_SIZE = 2  # Keep last 2 exchanges (4 messages)

def chat_windowed(user_input):
    """Chat with a sliding window memory."""
    # Use only the last K exchanges for the prompt
    window = full_history[-(WINDOW_SIZE * 2):]

    response = convo_chain.invoke({"history": window, "input": user_input})

    # Store in full history
    full_history.append(HumanMessage(content=user_input))
    full_history.append(AIMessage(content=response))
    return response

# Conversation that tests memory limits
turns = [
    "My favorite color is blue. Remember that!",
    "What is 2 + 2?",
    "What is the capital of France?",
    "What is my favorite color?",  # Should be forgotten with window=2
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_windowed(msg)}")

print(f"\nFull history: {len(full_history)} messages")
print(f"Window sees:  {WINDOW_SIZE * 2} messages")
print("Earlier context (like favorite color) may be forgotten!")


Turn 1:
  User: My favorite color is blue. Remember that!
  AI:   I've taken note: your favorite color is blue.

Turn 2:
  User: What is 2 + 2?
  AI:   The answer is 4.

Turn 3:
  User: What is the capital of France?
  AI:   Paris.

Turn 4:
  User: What is my favorite color?
  AI:   I don't know, can you tell me?

Full history: 8 messages
Window sees:  4 messages
Earlier context (like favorite color) may be forgotten!


### Experiment 5C: Summary Memory — Summarize Old Context

For long conversations: summarize older messages with the LLM, keep recent ones verbatim.

In [32]:
# Summary memory — summarize old messages, keep recent ones verbatim

summary_history = []
conversation_summary = ""
RECENT_WINDOW = 2  # Keep last 2 exchanges verbatim

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You write conversation summaries. "
     "Update the existing summary by incorporating the new messages.\n"
     "Current summary: {existing_summary}\n\n"
     "New messages to incorporate:\n{new_messages}\n\n"
     "Write the updated summary in plain prose. Under 30 words."),
    ("human", "Update the summary.")
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

summary_convo_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful AI tutor who gives short, direct answers. "
     "Conversation so far: {summary}\n\n"
     "Your response must be under 30 words."),
    MessagesPlaceholder(variable_name="recent"),
    ("human", "{input}")
])
summary_convo_chain = summary_convo_prompt | llm | StrOutputParser()

def chat_with_summary(user_input):
    global conversation_summary

    # Get recent messages for the window
    recent = summary_history[-(RECENT_WINDOW * 2):]

    # Generate response
    response = summary_convo_chain.invoke({
        "summary": conversation_summary or "(No prior conversation)",
        "recent": recent,
        "input": user_input
    })

    # Save to history
    summary_history.append(HumanMessage(content=user_input))
    summary_history.append(AIMessage(content=response))

    # Summarize old messages when history grows beyond window
    if len(summary_history) > RECENT_WINDOW * 2:
        old_messages = summary_history[:-(RECENT_WINDOW * 2)]
        new_msgs_text = "\n".join(f"{m.type}: {m.content}" for m in old_messages[-2:])
        conversation_summary = summarize_chain.invoke({
            "existing_summary": conversation_summary or "(none)",
            "new_messages": new_msgs_text
        })

    return response

# Test with a longer conversation
turns = [
    "My name is Alex and I love Python.",
    "I'm learning about chains in LangChain.",
    "What are the four document processing strategies?",
    "What's my name and what am I learning?",  # Tests if summary captures early context
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_with_summary(msg)}")

print(f"\nConversation summary: {conversation_summary}")
print("Summary memory compresses old context — scales to long conversations!")


Turn 1:
  User: My name is Alex and I love Python.
  AI:   Hi Alex, nice to meet you! Enjoy exploring the world of Python, and maybe even learning about sentence transformer models someday!

Turn 2:
  User: I'm learning about chains in LangChain.
  AI:   LangChain is a powerful library for building chain-based workflows in Python. Are you exploring how to use Chains to automate tasks or build custom integrations?

Turn 3:
  User: What are the four document processing strategies?
  AI:   In LangChain, the four main document processing strategies are:

1. `TextClassifier`
2. `SentimentIntensityAnalyzer`
3. `DocTuring`
4. `DocumentSearcher`

Turn 4:
  User: What's my name and what am I learning?
  AI:   Your name is Alex, and you're learning about sentence transformer models in Python.

Conversation summary: Alex, a Python enthusiast, is interested in learning about sentence transformer models, which transform sentences into numerical vectors to capture semantic meaning. Meanwhile, anoth

### Experiment 5D: Conversational RAG — Memory + Retrieval

The most powerful conversational pattern. Combines memory with retrieval AND **question reformulation** to handle follow-up questions.

**Why reformulation matters:**
- Turn 1: "What is LangChain?" → Clear query
- Turn 2: "Who created it?" → "it" refers to LangChain — retriever won't find docs!
- Reformulated: "Who created LangChain?" → Now the retriever works

In [33]:
# Simulated knowledge base
knowledge_base = {
    "chain": "Chains connect components into reusable pipelines. Types include sequential, parallel, conditional, and map-reduce.",
    "sequential": "Sequential chains process data through a linear pipeline where each step transforms data for the next step.",
    "parallel": "Parallel chains execute multiple operations simultaneously, providing 2-4x speedup for independent tasks.",
    "map-reduce": "Map-reduce processes each document independently (map phase) then combines results (reduce phase).",
    "memory": "Memory types include buffer (full history), window (last K), summary (LLM-compressed), and summary buffer.",
    "langchain": "LangChain was created by Harrison Chase in October 2022 as a framework for LLM applications.",
}

def retrieve(query: str) -> str:
    query_lower = query.lower()
    matched = [doc for key, doc in knowledge_base.items() if key in query_lower]
    return " ".join(matched) if matched else "No relevant documents found."

# Question reformulation chain
reformulate_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You rephrase follow-up questions into standalone questions. "
         "Given the conversation history, rewrite the question so it makes sense "
         "without the conversation context. "
         "If it is already standalone, return it unchanged. "
         "Respond with ONLY the rephrased question. Nothing else."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Answer chain
answer_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You answer questions based on the provided context. "
         "If the context does not contain the answer, say so. "
         "Your response must be under 40 words.\n\n"
         "Context: {context}"),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Conversational RAG with reformulation
rag_history = []

def convo_rag(user_question):
    # Step 1: Reformulate using history
    standalone = reformulate_chain.invoke({
        "chat_history": rag_history,
        "question": user_question
    })

    # Step 2: Retrieve with reformulated query
    context = retrieve(standalone)

    # Step 3: Answer
    answer = answer_chain.invoke({"context": context, "question": standalone})

    # Step 4: Update history
    rag_history.append(HumanMessage(content=user_question))
    rag_history.append(AIMessage(content=answer))

    return standalone, answer

# Test conversational RAG
turns = [
    "What are chains in LangChain?",
    "What types are there?",           # "there" = chains
    "Tell me about the parallel type.", # refers to parallel chains
    "How does memory work with them?",  # "them" = chains
]

for i, q in enumerate(turns, 1):
    standalone, answer = convo_rag(q)
    print(f"\nTurn {i}:")
    print(f"  Original:     {q}")
    print(f"  Reformulated: {standalone}")
    print(f"  Answer:       {answer}")


Turn 1:
  Original:     What are chains in LangChain?
  Reformulated: Can a chain be used to represent a specific sequence of tasks or operations?
  Answer:       Yes, a chain can be used to represent a specific sequence of tasks or operations. In the context of data processing pipelines, chains are often composed of multiple nodes that represent individual tasks or operations, connected together in a specific order. Each node in the chain receives input from the previous node and produces output for the next node, allowing for a linear execution of the sequence of tasks.

In this sense, a chain can be used to:

* Represent workflows or business processes
* Model data processing pipelines with multiple stages
* Define sequences of operations that need to be executed in a specific order

Turn 2:
  Original:     What types are there?
  Reformulated: In LangChain, chains can be categorized into several types:

1. **Manual Chain**: A manual chain is created by specifying each node individ

---

## 6. Transformation Chains — Custom Processing Logic

Use `RunnableLambda` to inject custom Python functions at any point in a chain.

| Position | Use Cases |
|----------|-----------|
| **Pre-processing** | Clean input, validate, truncate, normalize |
| **Post-processing** | Format output, filter sensitive data, add metadata |
| **Inter-step** | Transform between chain stages, enrich with lookups |

### Experiment 6A: Pre-Processing + Post-Processing Chain

In [34]:
import re
from datetime import datetime

# Pre-processing: clean and validate input
def preprocess(input_dict: dict) -> dict:
    question = input_dict["question"].strip()
    # Remove excessive whitespace
    question = re.sub(r'\s+', ' ', question)
    # Ensure it ends with a question mark
    if not question.endswith('?'):
        question += '?'
    return {"question": question}

# Post-processing: add metadata
def postprocess(response: str) -> dict:
    return {
        "answer": response,
        "timestamp": datetime.now().isoformat(),
        "word_count": len(response.split()),
        "model": MODEL
    }

# Full chain: preprocess → prompt → model → parse → postprocess
processed_chain = (
    RunnableLambda(preprocess)
    | ChatPromptTemplate.from_messages([
        ("system",
         "You are a concise technical assistant. "
         "Give a direct answer in plain prose. Under 30 words."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(postprocess)
)

# Messy input gets cleaned
result = processed_chain.invoke({"question": "   what  are   chains   in   langchain   "})

print("Result with metadata:")
for key, value in result.items():
    print(f"  {key}: {value}")

Result with metadata:
  answer: In LangChain, a "chain" refers to a sequence of tasks that are executed one after another, where each task builds upon the output of the previous one.

A chain can be thought of as a pipeline of computations, where each node in the chain represents a task or a model, and the edges between nodes represent the input/output relationships between them.

Here's an example of how chains work:

* Suppose you have two models: `model1` that takes text as input and outputs a vector representation, and `model2` that takes this vector representation as input and outputs a classification label.
* You can create a chain by connecting these two models in sequence: `input -> model1 -> output -> model2 -> classification_label`.
* When you run the chain, the first model processes the input text, producing an output vector. This output vector is then fed into the second model, which produces a classification label.
* The resulting chain can be represented as a directed gra

---

## 7. Chain Composition — Building Complex Systems

Chains can **contain other chains** as components. Build complex systems from simple, tested building blocks.

### Experiment 7A: Chain of Chains — Content Creation Pipeline

```python
research_chain = research_prompt | model | parser
outline_chain  = outline_prompt  | model | parser
writing_chain  = writing_prompt  | model | parser

full_pipeline = (
    RunnablePassthrough.assign(research=research_chain)
    | RunnablePassthrough.assign(outline=outline_chain)
    | writing_chain
)
```

In [35]:
# Content creation pipeline: research → outline → draft

# Sub-chain 1: Research
research_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a researcher who lists key facts. "
         "List exactly 3 facts about the topic. One line each, under 15 words per line. "
         "Format:\n1. Fact one\n2. Fact two\n3. Fact three\n"
         "Write nothing else."),
        ("human", "Research: {topic}")
    ]) | llm | StrOutputParser()
)

# Sub-chain 2: Outline (uses topic + research)
outline_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You create article outlines. "
         "Based on this research:\n{research}\n\n"
         "Create a 3-point outline. One line per point, under 10 words each. "
         "Format:\nI. Point one\nII. Point two\nIII. Point three\n"
         "Write nothing else."),
        ("human", "Topic: {topic}")
    ]) | llm | StrOutputParser()
)

# Sub-chain 3: Draft (uses topic + research + outline)
draft_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You write short technical articles. "
         "Write a single paragraph based on this research and outline. "
         "Research: {research}\n"
         "Outline: {outline}\n\n"
         "Your paragraph must be under 80 words. No bullet points, no headings."),
        ("human", "Topic: {topic}")
    ]) | llm | StrOutputParser()
)

# Compose into a full pipeline with context accumulation
content_pipeline = (
    RunnablePassthrough.assign(research=research_chain)
    | RunnablePassthrough.assign(outline=outline_chain)
    | RunnablePassthrough.assign(draft=draft_chain)
)

result = content_pipeline.invoke({"topic": "Document processing chains in LangChain"})

print("CONTENT CREATION PIPELINE")
print("=" * 50)

print("\nRESEARCH:")
print(result["research"])

print("\nOUTLINE:")
print(result["outline"])

print("\nDRAFT:")
display(Markdown(result["draft"]))

CONTENT CREATION PIPELINE

RESEARCH:
1. LangChain is an open-source Python library that enables flexible and efficient document processing workflows.
2. A key feature of LangChain is its support for document processing chains, which allow users to define complex workflows composed of multiple tasks, such as text classification, entity extraction, and image processing.
3. Document processing chains in LangChain can be defined using a simple and intuitive API, allowing developers to easily create custom workflows that meet their specific needs.

OUTLINE:
I. Define document processing chains using LangChain API
II. Compose complex workflows using multiple tasks and tools
III. Integrate with other NLP libraries for enhanced functionality

DRAFT:


Document processing chains in LangChain enable flexible and efficient workflows by allowing users to define complex sequences of tasks, such as text classification, entity extraction, and image processing. Using a simple and intuitive API, developers can compose these tasks into customized workflows that meet their specific needs, streamlining document processing tasks and reducing the need for manual scripting or integration with multiple tools.

### Experiment 7B: Self-Correcting Chain

The chain checks its own output and retries if quality is insufficient.

```
Input → Generate → Validate → Pass? → Return
                       ↓ Fail
              Regenerate with feedback → Validate again
```

In [36]:
# Self-correcting chain with one retry

# Generation chain
generate_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful assistant who answers questions accurately. "
         "You must include at least one specific example in your answer. "
         "Your response must be under 60 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
)

# Validation chain — checks if the answer meets quality criteria
validate_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a quality checker. Check if this answer:\n"
         "1. Answers the question directly\n"
         "2. Contains at least one specific example\n\n"
         "Respond with ONLY valid JSON in this exact format:\n"
         '{{"pass": true, "feedback": "looks good"}}\n'
         "or\n"
         '{{"pass": false, "feedback": "missing example"}}\n'
         "No markdown, no explanation, only the JSON object."),
        ("human", "Question: {question}\nAnswer: {answer}")
    ]) | llm | JsonOutputParser()
)

# Regeneration chain — uses feedback to improve
regenerate_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Your previous answer was rejected. "
         "Rewrite the answer addressing this feedback: {feedback}\n"
         "Previous answer: {answer}\n\n"
         "Include a specific example. Under 60 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
)


def self_correcting_invoke(question: str, max_attempts: int = 2) -> str:
    """Generate, validate, and retry if needed."""
    answer = generate_chain.invoke({"question": question})

    for attempt in range(max_attempts):
        print(f"  Attempt {attempt + 1}: {answer[:80]}...")

        try:
            validation = validate_chain.invoke({
                "question": question,
                "answer": answer
            })

            if validation.get("pass", False):
                print(f"  Validation: PASSED")
                return answer

            feedback = validation.get("feedback", "Improve the answer.")
            print(f"  Validation: FAILED — {feedback}")

            # Regenerate with feedback
            answer = regenerate_chain.invoke({
                "question": question,
                "answer": answer,
                "feedback": feedback
            })
        except Exception as e:
            print(f"  Validation parse error: {e}")
            return answer

    return answer


question = "What are sequential chains in LangChain?"
print(f"Q: {question}\n")

final = self_correcting_invoke(question)

print(f"\n{'=' * 50}")
print("FINAL ANSWER:")
print(f"{'=' * 50}")
display(Markdown(final))

Q: What are sequential chains in LangChain?

  Attempt 1: In LangChain, a Sequential Chain is a data structure that represents a sequence ...
  Validation: PASSED

FINAL ANSWER:


In LangChain, a Sequential Chain is a data structure that represents a sequence of tasks or operations that can be executed in a pipeline-like fashion.

A Sequential Chain is composed of multiple steps, each of which performs a specific task or operation. These steps can be thought of as "tasks" or "actions" that are chained together to achieve a larger goal.

Here's an example of how you might define a Sequential Chain using LangChain:
```python
from langchain import LLMChain

# Define the tasks in our chain
task1 = lambda text: text.upper()
task2 = lambda text: text[::-1]

# Create a Sequential Chain with these tasks
chain = LLMChain(
    task1,  # first task: convert to uppercase
    task2  # second task: reverse the string
)

# Execute the chain on some input text
input_text = "hello world"
output_text = chain.execute(input_text)
print(output_text)  # prints "DLROW OLLEH"
```
In this example, we define two tasks (`task1` and `task2`) that take a string as input. The first task converts the string to uppercase, while the second task reverses the string.

We then create a Sequential Chain with these two tasks, and execute it on some input text using the `execute()` method.

The output of the chain is the result of applying both tasks in sequence: first converting to uppercase, and then reversing the string.

---

## 8. Chain Design Best Practices & Anti-Patterns

### Best Practices

| Practice | Description |
|----------|-------------|
| Start simple, add incrementally | Begin with `prompt \| model \| parser` — add steps only when needed |
| One responsibility per step | Each component does exactly one thing |
| Test sub-chains independently | Build and test each sub-chain before composing |
| Use `assign()` for context | Don't lose information between steps |
| Add fallbacks at model level | `.with_fallbacks()` for provider outages |
| Name chains for observability | `.with_config(run_name="name")` for LangSmith |
| Separate prompts from logic | Templates as standalone objects, not inline strings |
| Stream for user-facing chains | Better perceived performance |

### Anti-Patterns to Avoid

| Anti-Pattern | Problem | Better Approach |
|-------------|---------|----------------|
| **God Chain** | Single massive chain doing everything | Break into focused sub-chains |
| **Prompt Stuffing** | Cramming too much into one prompt | Multi-step chains with focused prompts |
| **No Error Handling** | No fallbacks or retries | `.with_retry()` + `.with_fallbacks()` |
| **Hardcoded Models** | Model names embedded in code | Use `configurable_alternatives()` |
| **Sequential for Parallel** | Running independent ops one by one | Use `RunnableParallel` |
| **No Observability** | Chains without tracing | Enable LangSmith + name every chain |

---

## 9. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Build your own chain!
# ============================================================

# Choose a chain type to try
chain_type = "sequential"  # Options: "sequential", "parallel", "conditional"

# ============================================================

if chain_type == "sequential":
    # Sequential: generate → critique
    step1 = ChatPromptTemplate.from_messages([
        ("system",
         "You write concise technical explanations. "
         "Explain in plain prose. Under 30 words."),
        ("human", "{topic}")
    ]) | llm | StrOutputParser()

    step2 = ChatPromptTemplate.from_messages([
        ("system",
         "You critique technical explanations. "
         "Identify one thing that is missing. "
         "Your response must be a single sentence, under 20 words."),
        ("human", "{text}")
    ]) | llm | StrOutputParser()

    explanation = step1.invoke({"topic": "Chains in LangChain"})
    critique = step2.invoke({"text": explanation})
    print(f"Explanation: {explanation}")
    print(f"\nCritique: {critique}")

elif chain_type == "parallel":
    # Parallel: 3 perspectives at once
    par = RunnableParallel(
        beginner=make_chain(
            "You explain for beginners using a simple analogy. One sentence, under 20 words."
        ),
        expert=make_chain(
            "You give expert-level technical definitions. One sentence, under 20 words."
        ),
        analogy=make_chain(
            "You explain concepts using a cooking analogy. One sentence, under 20 words."
        ),
    )
    result = par.invoke({"topic": "Map-Reduce chains"})
    for key, val in result.items():
        print(f"{key}: {val}\n")

elif chain_type == "conditional":
    # Conditional: route by question type
    branch = RunnableBranch(
        (lambda x: "?" in x["question"],
         make_chain("You answer questions directly. Under 30 words.")),
        make_chain("You respond to statements with a short comment. Under 30 words.")
    )
    print(branch.invoke({"topic": "What is a chain?", "question": "What is a chain?"}))

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **Chains** | Connect components into complete, reusable workflows via LCEL |
| **Sequential** | Linear pipeline: A → B → C — most common pattern |
| **Parallel** | Simultaneous branches — 2-4x speedup for independent tasks |
| **Conditional** | Route by condition — keyword-based or LLM-classified |
| **Context Accumulation** | `assign()` chains carry ALL previous outputs forward |
| **Stuff** | All docs in one prompt — simple, fast, limited by context window |
| **Map-Reduce** | Process each doc → combine — scales to unlimited docs |
| **Refine** | Sequential refinement — highest quality, slowest |
| **Map-Rerank** | Score each → pick best — great for factual Q&A |
| **Buffer Memory** | Full history — simple, grows unbounded |
| **Window Memory** | Last K turns — fixed size, may forget early context |
| **Summary Memory** | LLM-compressed history — scales to long conversations |
| **Conversational RAG** | Memory + retrieval + question reformulation |
| **Self-Correcting** | Generate → validate → retry — quality assurance |
| **Chain Composition** | Chains contain chains — build complex from simple |